# 🎯 Multi-Object Detection & Persistent ID Tracking
**YOLOv8n + DeepSORT Pipeline**

Run each cell top-to-bottom in Google Colab (Runtime → Run All).

## Step 1 — Install Dependencies

In [ ]:
!pip install ultralytics deep-sort-realtime opencv-python-headless -q
print('✅ Installation complete')

## Step 2 — Upload Your Video

Upload a sports/event video from your device, OR paste a YouTube URL below.

In [ ]:
# Option A: Upload from your computer
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f'Uploaded: {VIDEO_PATH}')

In [ ]:
# Option B: Download from YouTube (uncomment to use)
# !pip install yt-dlp -q
# YOUTUBE_URL = 'https://www.youtube.com/watch?v=YOUR_VIDEO_ID'
# !yt-dlp -f 'best[height<=480]' -o 'input_video.mp4' {YOUTUBE_URL}
# VIDEO_PATH = 'input_video.mp4'

## Step 3 — Run the Pipeline

In [ ]:
import cv2, time, numpy as np
from collections import defaultdict
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ── Configuration ────────────────────────────────────────────────────────────
OUTPUT_PATH     = 'output_tracked.mp4'
CONF_THRESHOLD  = 0.40   # lower = more detections (but more false positives)
SKIP_FRAMES     = 1      # 1 = process every frame; 2 = every other frame (faster)
MAX_AGE         = 30     # frames to keep a lost track alive

# ── Colour palette ────────────────────────────────────────────────────────────
COLORS = [
    (0,255,0),(255,0,0),(0,0,255),(255,255,0),
    (0,255,255),(255,0,255),(128,255,0),(0,128,255),
]
def get_color(tid): return COLORS[tid % len(COLORS)]

# ── Load models ───────────────────────────────────────────────────────────────
print('[INFO] Loading YOLOv8n ...')
model   = YOLO('yolov8n.pt')
tracker = DeepSort(max_age=MAX_AGE, n_init=3,
                   nms_max_overlap=1.0, max_cosine_distance=0.3,
                   embedder='mobilenet', half=True, bgr=True)

# ── Open video ────────────────────────────────────────────────────────────────
cap    = cv2.VideoCapture(VIDEO_PATH)
W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS    = cap.get(cv2.CAP_PROP_FPS) or 30.0
TOTAL  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))

print(f'[INFO] {W}x{H} @ {FPS:.1f}fps  |  {TOTAL} frames')

# ── State ─────────────────────────────────────────────────────────────────────
trajectories    = defaultdict(list)
count_log       = []       # (frame_no, active_count)
all_ids         = set()
frame_no        = 0

# ── Main loop ─────────────────────────────────────────────────────────────────
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_no += 1

    if frame_no % SKIP_FRAMES != 0:
        out.write(frame); continue

    # Detect persons
    results    = model(frame, classes=[0], conf=CONF_THRESHOLD, verbose=False)[0]
    detections = []
    for box in results.boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        detections.append(([x1,y1,x2-x1,y2-y1], float(box.conf[0]), 'person'))

    # Track
    tracks     = tracker.update_tracks(detections, frame=frame)
    active     = 0

    for track in tracks:
        if not track.is_confirmed(): continue
        tid  = track.track_id
        l,t,r,b = map(int, track.to_ltrb())
        all_ids.add(tid); active += 1
        cx, cy = (l+r)//2, (t+b)//2
        trajectories[tid].append((cx, cy))

        color = get_color(tid)
        # Box
        cv2.rectangle(frame,(l,t),(r,b),color,2)
        # Label
        lbl = f'ID:{tid}'
        (lw,lh),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX,0.6,2)
        cv2.rectangle(frame,(l,t-lh-10),(l+lw+4,t),color,-1)
        cv2.putText(frame,lbl,(l+2,t-5),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,0),2)
        # Trajectory
        pts = trajectories[tid][-30:]
        for i in range(1,len(pts)):
            if pts[i-1] and pts[i]:
                cv2.line(frame, pts[i-1], pts[i], color, 2)

    # HUD
    cv2.rectangle(frame,(0,0),(240,60),(0,0,0),-1)
    cv2.putText(frame,f'Frame:{frame_no}',(6,20),cv2.FONT_HERSHEY_SIMPLEX,0.5,(200,200,200),1)
    cv2.putText(frame,f'Tracked:{active}',(6,42),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,128),1)

    count_log.append((frame_no, active))
    out.write(frame)

    if frame_no % 100 == 0:
        print(f'  Frame {frame_no}/{TOTAL}  active={active}  all_ids={len(all_ids)}')

cap.release(); out.release()
print(f'\n✅ Done! Unique IDs: {len(all_ids)}  →  {OUTPUT_PATH}')

## Step 4 — Save Screenshots & Chart

In [ ]:
import matplotlib.pyplot as plt, os
os.makedirs('screenshots', exist_ok=True)

# Count-over-time chart
frames_list  = [x[0] for x in count_log]
counts_list  = [x[1] for x in count_log]
plt.figure(figsize=(10,4))
plt.plot(frames_list, counts_list, color='lime', linewidth=1.5)
plt.fill_between(frames_list, counts_list, alpha=0.2, color='lime')
plt.xlabel('Frame'); plt.ylabel('Active Tracked Objects')
plt.title('Object Count Over Time'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('screenshots/count_over_time.png', dpi=120)
plt.show()
print('Chart saved!')

In [ ]:
# Extract sample screenshots from output video
cap2  = cv2.VideoCapture(OUTPUT_PATH)
TOTAL2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
for i, target in enumerate([int(TOTAL2*0.1), int(TOTAL2*0.3),
                              int(TOTAL2*0.6), int(TOTAL2*0.9)]):
    cap2.set(cv2.CAP_PROP_POS_FRAMES, target)
    ret, frm = cap2.read()
    if ret:
        path = f'screenshots/frame_{i+1}.jpg'
        cv2.imwrite(path, frm)
        print(f'Saved {path}')
cap2.release()
print('✅ Screenshots saved in screenshots/')

## Step 5 — Download Output

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)
for i in range(1,5):
    try: files.download(f'screenshots/frame_{i}.jpg')
    except: pass
files.download('screenshots/count_over_time.png')
print('✅ All files downloaded!')